In [ ]:
# Cell 1: Install dependencies
!pip install -q scanpy anndata igraph leidenalg scikit-learn scipy requests
!pip install -q cellxgene-census

In [ ]:
# Cell 2: Mount Drive and upload project files
from google.colab import drive
drive.mount('/content/drive')

RESULTS_DIR     = '/content/drive/MyDrive/CellJEPA_results/robustness/'
SIGREG_CKPT_DIR = '/content/drive/MyDrive/CellJEPA_results/kidney_sigreg/'

import os
os.makedirs(RESULTS_DIR, exist_ok=True)

# Upload all .py files to /content/ before running cells below:
# cell_jepa.py, cell_sigreg.py, losses.py, preprocessing.py,
# trainer.py, metrics.py, compare_pbmc3k.py, run_ablation.py,
# run_transfer.py, run_multiseed.py, run_lr_sweep.py
SIG_CKPT  = os.path.join(SIGREG_CKPT_DIR, 'kidney_sigreg_final.pt')
SIG_GENES = os.path.join(SIGREG_CKPT_DIR, 'kidney_sigreg_gene_names.json')

import json
print('Drive mounted.')
print('SIGReg checkpoint:', '\u2713' if os.path.exists(SIG_CKPT) else '\u2717 MISSING', SIG_CKPT)
print('SIGReg genes:     ', '\u2713' if os.path.exists(SIG_GENES) else '\u2717 MISSING', SIG_GENES)
if os.path.exists(SIG_GENES):
    sample = json.load(open(SIG_GENES))[:3]
    looks_ensembl = all(g.startswith('ENSG') for g in sample)
    print(f'  Gene name sample: {sample}  {"\u26a0 STILL ENSEMBL - run fix_kidney_gene_names.py" if looks_ensembl else "\u2713 symbols OK"}')
print('Results will be saved to:', RESULTS_DIR)

In [ ]:
# Cell 3: Smoke test — both scripts, 1 seed, 1 LR scale, 200 cells, 1 epoch (~5 min CPU)
import subprocess, os

REPO_DIR = '/content'

print('=== Smoke test: run_multiseed.py ===')
result = subprocess.run(
    ['python3', '-u', os.path.join(REPO_DIR, 'run_multiseed.py'),
     '--smoke_test', '--device', 'cpu',
     '--seeds', '42',
     '--sigreg_checkpoint', SIG_CKPT,
     '--sigreg_genes',      SIG_GENES,
     '--results_file', 'results_multiseed_smoke.txt'],
    capture_output=True, text=True
)
print(result.stdout)
if result.stderr:
    print('STDERR:', result.stderr[-2000:])

print('\n=== Smoke test: run_lr_sweep.py ===')
result2 = subprocess.run(
    ['python3', '-u', os.path.join(REPO_DIR, 'run_lr_sweep.py'),
     '--smoke_test', '--device', 'cpu',
     '--lr_scales', '1.0',
     '--sigreg_checkpoint', SIG_CKPT,
     '--sigreg_genes',      SIG_GENES,
     '--results_file', 'results_lr_sweep_smoke.txt'],
    capture_output=True, text=True
)
print(result2.stdout)
if result2.stderr:
    print('STDERR:', result2.stderr[-2000:])

In [ ]:
# Cell 4: Full multi-seed experiment — 3 seeds × 2 conditions
import subprocess, time, threading, os

REPO_DIR = '/content'

t0 = time.time()
proc = subprocess.Popen(
    ['python3', '-u', os.path.join(REPO_DIR, 'run_multiseed.py'),
     '--device', 'cuda',
     '--seeds', '42', '123', '999',
     '--pretrain_epochs', '4',
     '--finetune_epochs', '30',
     '--sigreg_checkpoint', SIG_CKPT,
     '--sigreg_genes',      SIG_GENES,
     '--results_file', 'results_multiseed.txt'],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True, bufsize=1
)

def stream(pipe):
    for line in pipe:
        print(line, end='', flush=True)

t_out = threading.Thread(target=stream, args=(proc.stdout,))
t_err = threading.Thread(target=stream, args=(proc.stderr,))
t_out.start(); t_err.start()
t_out.join(); t_err.join()

rc = proc.wait()
if rc != 0:
    print(f'\n*** PROCESS EXITED WITH CODE {rc} — see stderr above ***')
else:
    print(f'\nDone in {(time.time()-t0)/60:.1f} min')

In [ ]:
# Cell 5: LR sweep — 3 scales for SIGReg kidney fine-tuning
import subprocess, time, threading, os

REPO_DIR = '/content'

t0 = time.time()
proc = subprocess.Popen(
    ['python3', '-u', os.path.join(REPO_DIR, 'run_lr_sweep.py'),
     '--device', 'cuda',
     '--lr_scales', '0.1', '0.3', '1.0',
     '--pretrain_epochs', '4',
     '--finetune_epochs', '30',
     '--sigreg_checkpoint', SIG_CKPT,
     '--sigreg_genes',      SIG_GENES,
     '--results_file', 'results_lr_sweep.txt'],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True, bufsize=1
)

def stream(pipe):
    for line in pipe:
        print(line, end='', flush=True)

t_out = threading.Thread(target=stream, args=(proc.stdout,))
t_err = threading.Thread(target=stream, args=(proc.stderr,))
t_out.start(); t_err.start()
t_out.join(); t_err.join()

rc = proc.wait()
if rc != 0:
    print(f'\n*** PROCESS EXITED WITH CODE {rc} — see stderr above ***')
else:
    print(f'\nDone in {(time.time()-t0)/60:.1f} min')

In [ ]:
# Cell 6: Display results and plots
import os, re
import numpy as np
import matplotlib.pyplot as plt

for fname in ['results_multiseed.txt', 'results_lr_sweep.txt']:
    if os.path.exists(fname):
        print(f'--- {fname} ---')
        print(open(fname).read())

# ---- Plot 1: Multi-seed bar chart with error bars ----
def parse_multiseed(path):
    if not os.path.exists(path):
        return {}
    text = open(path).read()
    sections = re.split(r'Zero-shot|Fine-tuned', text)
    data = {'SIGReg (scratch)': {}, 'SIGReg (kidney)': {}}
    phase_keys = ['zero_shot', 'fine_tuned']
    for i, phase in enumerate(phase_keys):
        if i + 1 >= len(sections):
            break
        for cond in data:
            data[cond][phase] = {'nmi': [], 'ari': [], 'asw': [], 'avg_bio': []}
        for line in sections[i + 1].splitlines():
            if 'MEAN' in line:
                continue
            nums = re.findall(r'\d+\.\d{4}', line)
            if len(nums) == 4:
                for cond in data:
                    if cond.replace(' ', '') in line.replace(' ', ''):
                        data[cond][phase]['nmi'].append(float(nums[0]))
                        data[cond][phase]['ari'].append(float(nums[1]))
                        data[cond][phase]['asw'].append(float(nums[2]))
                        data[cond][phase]['avg_bio'].append(float(nums[3]))
    return data

ms_data = parse_multiseed('results_multiseed.txt')
if ms_data and any(ms_data[c].get('fine_tuned', {}).get('avg_bio') for c in ms_data):
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    conditions = list(ms_data.keys())
    colors = ['#4C72B0', '#DD8452']
    for ax, (phase, plabel) in zip(axes, [('zero_shot', 'Zero-shot'), ('fine_tuned', 'Fine-tuned')]):
        x = np.arange(len(conditions))
        for ci, (cond, col) in enumerate(zip(conditions, colors)):
            vals = ms_data[cond].get(phase, {}).get('avg_bio', [])
            if vals:
                mu, sd = np.mean(vals), np.std(vals)
                ax.bar(x[ci], mu, 0.5, yerr=sd, color=col, alpha=0.85,
                       capsize=6, label=cond)
                ax.text(x[ci], mu + sd + 0.01, f'{mu:.3f}\u00b1{sd:.3f}',
                        ha='center', fontsize=8)
        ax.set_xticks(x)
        ax.set_xticklabels(conditions, fontsize=9)
        ax.set_title(f'{plabel} AvgBIO (mean \u00b1 std, n=3 seeds)')
        ax.set_ylabel('AvgBIO')
        ax.set_ylim(0, 1.0)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.yaxis.grid(True, linestyle='--', alpha=0.4)
        ax.set_axisbelow(True)
    fig.suptitle('SIGReg Multi-Seed Robustness', fontsize=11)
    plt.tight_layout()
    plt.savefig('multiseed_results.png', dpi=150)
    plt.show()
    print('Saved multiseed_results.png')

# ---- Plot 2: LR sweep line plot ----
def parse_lr_sweep(path):
    if not os.path.exists(path):
        return {}
    text = open(path).read()
    sections = re.split(r'Zero-shot|Fine-tuned', text)
    data = {}
    phase_keys = ['zero_shot', 'fine_tuned']
    for i, phase in enumerate(phase_keys):
        if i + 1 >= len(sections):
            break
        for line in sections[i + 1].splitlines():
            nums = re.findall(r'\d+\.\d{4}', line)
            if len(nums) == 4:
                name = line[:36].strip()
                if name and not name.startswith(('-', '=', 'C')):
                    if name not in data:
                        data[name] = {}
                    data[name][phase] = {
                        'nmi': float(nums[0]), 'ari': float(nums[1]),
                        'asw': float(nums[2]), 'avg_bio': float(nums[3])
                    }
    return data

lr_data = parse_lr_sweep('results_lr_sweep.txt')
if lr_data:
    kidney_keys = [k for k in lr_data if 'kidney' in k]
    scratch_keys = [k for k in lr_data if 'scratch' in k]

    def extract_lr(name):
        # Match scientific notation, stopping before closing paren e.g. lr=1.0e-04)
        m = re.search(r'lr=([0-9eE.+-]+)', name)
        return float(m.group(1)) if m else None

    kidney_keys_sorted = sorted(kidney_keys, key=extract_lr)
    lrs = [extract_lr(k) for k in kidney_keys_sorted]
    ft_avbio = [lr_data[k].get('fine_tuned', {}).get('avg_bio', 0) for k in kidney_keys_sorted]

    scratch_ft = lr_data[scratch_keys[0]].get('fine_tuned', {}).get('avg_bio', 0) if scratch_keys else None

    fig, ax = plt.subplots(figsize=(7, 5))
    ax.plot(lrs, ft_avbio, 'o-', color='#DD8452', linewidth=2, markersize=8, label='SIGReg (kidney)')
    if scratch_ft:
        ax.axhline(scratch_ft, color='#4C72B0', linestyle='--', linewidth=1.5,
                   label=f'SIGReg (scratch) = {scratch_ft:.3f}')
    for lr, val in zip(lrs, ft_avbio):
        ax.annotate(f'{val:.3f}', (lr, val), textcoords='offset points',
                    xytext=(0, 8), ha='center', fontsize=9)
    ax.set_xscale('log')
    ax.set_xlabel('Fine-tuning LR')
    ax.set_ylabel('Fine-tuned AvgBIO')
    ax.set_title('SIGReg Kidney Fine-Tuning LR Sweep\n(Collapse Diagnosis)')
    ax.set_ylim(0, 1.0)
    ax.legend(fontsize=9)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.yaxis.grid(True, linestyle='--', alpha=0.4)
    ax.set_axisbelow(True)
    plt.tight_layout()
    plt.savefig('lr_sweep_results.png', dpi=150)
    plt.show()
    print('Saved lr_sweep_results.png')

In [ ]:
# Cell 7: Save all results to Drive
import shutil, os

RESULTS_DIR = '/content/drive/MyDrive/CellJEPA_results/robustness/'
files = [
    'results_multiseed.txt', 'results_multiseed_smoke.txt', 'multiseed_results.png',
    'results_lr_sweep.txt',  'results_lr_sweep_smoke.txt',  'lr_sweep_results.png',
]
for f in files:
    if os.path.exists(f):
        shutil.copy(f, RESULTS_DIR)
        print(f'Copied {f}')
    else:
        print(f'Not found: {f} (skipping)')
print(f'Done. Files in {RESULTS_DIR}')

In [ ]:
# Cell 8: SIGReg weight sweep — smoke test (~5 min CPU)
import subprocess, os

REPO_DIR = '/content'

print('=== Smoke test: run_sigreg_reg_sweep.py ===')
result = subprocess.run(
    ['python3', '-u', os.path.join(REPO_DIR, 'run_sigreg_reg_sweep.py'),
     '--smoke_test', '--device', 'cpu',
     '--sigreg_weights', '0.5', '2.0',
     '--sigreg_checkpoint', SIG_CKPT,
     '--sigreg_genes',      SIG_GENES,
     '--results_file', 'results_sigreg_reg_sweep_smoke.txt'],
    capture_output=True, text=True
)
print(result.stdout)
if result.stderr:
    print('STDERR:', result.stderr[-2000:])

In [ ]:
# Cell 9: SIGReg weight sweep — full run (~40 min on A100)
# 1 scratch baseline + 4 kidney conditions × ~2.5 min each = ~12 min kidney
# + scratch pre-train ~25 min = ~37 min total
import subprocess, time, threading, os

REPO_DIR = '/content'

t0 = time.time()
proc = subprocess.Popen(
    ['python3', '-u', os.path.join(REPO_DIR, 'run_sigreg_reg_sweep.py'),
     '--device', 'cuda',
     '--sigreg_weights', '0.5', '2.0', '5.0', '10.0',
     '--pretrain_epochs', '4',
     '--finetune_epochs', '30',
     '--sigreg_checkpoint', SIG_CKPT,
     '--sigreg_genes',      SIG_GENES,
     '--results_file', 'results_sigreg_reg_sweep.txt'],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True, bufsize=1
)

def stream(pipe):
    for line in pipe:
        print(line, end='', flush=True)

t_out = threading.Thread(target=stream, args=(proc.stdout,))
t_err = threading.Thread(target=stream, args=(proc.stderr,))
t_out.start(); t_err.start()
t_out.join(); t_err.join()

rc = proc.wait()
if rc != 0:
    print(f'\n*** PROCESS EXITED WITH CODE {rc} — see stderr above ***')
else:
    print(f'\nDone in {(time.time()-t0)/60:.1f} min')

In [ ]:
# Cell 10: Display SIGReg weight sweep results and plot
import os, re
import numpy as np
import matplotlib.pyplot as plt

if os.path.exists('results_sigreg_reg_sweep.txt'):
    print(open('results_sigreg_reg_sweep.txt').read())

def parse_reg_sweep(path):
    if not os.path.exists(path):
        return {}
    text = open(path).read()
    sections = re.split(r'Zero-shot|Fine-tuned', text)
    data = {}
    phase_keys = ['zero_shot', 'fine_tuned']
    for i, phase in enumerate(phase_keys):
        if i + 1 >= len(sections):
            break
        for line in sections[i + 1].splitlines():
            nums = re.findall(r'\d+\.\d{4}', line)
            if len(nums) == 4:
                name = line[:44].strip()
                if name and not name.startswith(('-', '=', 'C')):
                    if name not in data:
                        data[name] = {}
                    data[name][phase] = {
                        'nmi': float(nums[0]), 'ari': float(nums[1]),
                        'asw': float(nums[2]), 'avg_bio': float(nums[3])
                    }
    return data

reg_data = parse_reg_sweep('results_sigreg_reg_sweep.txt')
if reg_data:
    kidney_keys = [k for k in reg_data if 'kidney' in k]
    scratch_keys = [k for k in reg_data if 'scratch' in k]

    def extract_w(name):
        m = re.search(r'w_sigreg=([0-9.]+)', name)
        return float(m.group(1)) if m else None

    kidney_keys_sorted = sorted(kidney_keys, key=extract_w)
    ws = [extract_w(k) for k in kidney_keys_sorted]
    ft_avbio = [reg_data[k].get('fine_tuned', {}).get('avg_bio', 0) for k in kidney_keys_sorted]
    zs_avbio = [reg_data[k].get('zero_shot', {}).get('avg_bio', 0) for k in kidney_keys_sorted]

    scratch_ft = reg_data[scratch_keys[0]].get('fine_tuned', {}).get('avg_bio', 0) if scratch_keys else None

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(ws, ft_avbio, 'o-', color='#DD8452', linewidth=2, markersize=8,
            label='SIGReg (kidney) fine-tuned')
    ax.plot(ws, zs_avbio, 's--', color='#DD8452', linewidth=1.5, markersize=6,
            alpha=0.5, label='SIGReg (kidney) zero-shot')
    if scratch_ft:
        ax.axhline(scratch_ft, color='#4C72B0', linestyle='--', linewidth=1.5,
                   label=f'SIGReg (scratch) fine-tuned = {scratch_ft:.3f}')
    for w, val in zip(ws, ft_avbio):
        ax.annotate(f'{val:.3f}', (w, val), textcoords='offset points',
                    xytext=(0, 8), ha='center', fontsize=9)
    ax.set_xlabel('w_sigreg during fine-tuning')
    ax.set_ylabel('AvgBIO')
    ax.set_title('SIGReg Kidney Fine-Tuning: Regularisation Weight Sweep\n(Collapse Diagnosis — Regularisation Strength vs. Geometry Mismatch)')
    ax.set_ylim(0, 1.0)
    ax.legend(fontsize=9)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.yaxis.grid(True, linestyle='--', alpha=0.4)
    ax.set_axisbelow(True)
    plt.tight_layout()
    plt.savefig('sigreg_reg_sweep_results.png', dpi=150)
    plt.show()
    print('Saved sigreg_reg_sweep_results.png')

In [ ]:
# Cell 11: Save SIGReg weight sweep results to Drive
import shutil, os

RESULTS_DIR = '/content/drive/MyDrive/CellJEPA_results/robustness/'
files = [
    'results_sigreg_reg_sweep.txt',
    'results_sigreg_reg_sweep_smoke.txt',
    'sigreg_reg_sweep_results.png',
]
for f in files:
    if os.path.exists(f):
        shutil.copy(f, RESULTS_DIR)
        print(f'Copied {f}')
    else:
        print(f'Not found: {f} (skipping)')
print(f'Done. Files in {RESULTS_DIR}')

In [ ]:
# Cell 12: PBMC-68K pre-training — smoke test (~3 min CPU)
import subprocess, os

REPO_DIR = '/content'

result = subprocess.run(
    ['python3', '-u', os.path.join(REPO_DIR, 'pretrain_pbmc68k.py'),
     '--smoke_test', '--device', 'cpu', '--cache_dir', '/content'],
    capture_output=True, text=True
)
print(result.stdout)
if result.stderr:
    print('STDERR:', result.stderr[-3000:])

In [ ]:
# Cell 13: PBMC-68K pre-training — full run (~60 min on A100)
# Pre-trains both Cell-JEPA and SIGReg on the canonical PBMC-68K dataset.
# Requires the tarball on Drive: fresh_68k_pbmc_donor_a_filtered_gene_bc_matrices.tar.gz
import subprocess, time, threading, os

REPO_DIR    = '/content'
RESULTS_DIR = '/content/drive/MyDrive/CellJEPA_results/pbmc68k_pretrain/'
TAR_PATH    = '/content/drive/MyDrive/fresh_68k_pbmc_donor_a_filtered_gene_bc_matrices.tar.gz'

t0 = time.time()
proc = subprocess.Popen(
    ['python3', '-u', os.path.join(REPO_DIR, 'pretrain_pbmc68k.py'),
     '--device', 'cuda',
     '--n_epochs', '4',
     '--batch_size', '32',
     '--tar_path', TAR_PATH,
     '--cache_dir', '/content',
     '--drive_dir', RESULTS_DIR],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True, bufsize=1
)

def stream(pipe):
    for line in pipe:
        print(line, end='', flush=True)

t_out = threading.Thread(target=stream, args=(proc.stdout,))
t_err = threading.Thread(target=stream, args=(proc.stderr,))
t_out.start(); t_err.start()
t_out.join(); t_err.join()

rc = proc.wait()
if rc != 0:
    print(f'
*** PROCESS EXITED WITH CODE {rc} ***')
else:
    print(f'
Done in {(time.time()-t0)/60:.1f} min')

In [ ]:
# Cell 14: PBMC-68K → PBMC-3K transfer — smoke test (~5 min CPU)
import subprocess, os

REPO_DIR    = '/content'
RESULTS_DIR = '/content/drive/MyDrive/CellJEPA_results/pbmc68k_pretrain/'

result = subprocess.run(
    ['python3', '-u', os.path.join(REPO_DIR, 'run_transfer_pbmc68k.py'),
     '--smoke_test', '--device', 'cpu',
     '--results_file', 'results_transfer_pbmc68k_smoke.txt'],
    capture_output=True, text=True
)
print(result.stdout)
if result.stderr:
    print('STDERR:', result.stderr[-3000:])

In [ ]:
# Cell 15: PBMC-68K → PBMC-3K transfer — full run (~90 min on A100)
# 4 conditions: (Cell-JEPA / SIGReg) × (scratch / PBMC-68K pre-trained)
# Requires Cell 13 to have completed first (checkpoint files must exist).
import subprocess, time, threading, os

REPO_DIR    = '/content'
RESULTS_DIR = '/content/drive/MyDrive/CellJEPA_results/pbmc68k_pretrain/'

JEPA_CKPT   = os.path.join(RESULTS_DIR, 'pbmc68k_jepa_final.pt')
SIGREG_CKPT = os.path.join(RESULTS_DIR, 'pbmc68k_sigreg_final.pt')
GENES_JSON  = os.path.join(RESULTS_DIR, 'pbmc68k_gene_names.json')

t0 = time.time()
proc = subprocess.Popen(
    ['python3', '-u', os.path.join(REPO_DIR, 'run_transfer_pbmc68k.py'),
     '--device', 'cuda',
     '--pretrain_epochs', '4',
     '--finetune_epochs', '30',
     '--jepa_checkpoint',   JEPA_CKPT,
     '--sigreg_checkpoint', SIGREG_CKPT,
     '--pbmc68k_genes',     GENES_JSON,
     '--results_file',      'results_transfer_pbmc68k.txt'],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True, bufsize=1
)

def stream(pipe):
    for line in pipe:
        print(line, end='', flush=True)

t_out = threading.Thread(target=stream, args=(proc.stdout,))
t_err = threading.Thread(target=stream, args=(proc.stderr,))
t_out.start(); t_err.start()
t_out.join(); t_err.join()

rc = proc.wait()
if rc != 0:
    print(f'
*** PROCESS EXITED WITH CODE {rc} ***')
else:
    print(f'
Done in {(time.time()-t0)/60:.1f} min')

In [ ]:
# Cell 16: Display PBMC-68K transfer results
import os, re
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

path = 'results_transfer_pbmc68k.txt'
if os.path.exists(path):
    print(open(path).read())

def parse_transfer_results(path):
    if not os.path.exists(path):
        return {}, {}
    zs, ft = {}, {}
    current = None
    for line in open(path):
        if 'Zero-shot' in line:  current = zs
        elif 'Fine-tuned' in line: current = ft
        m = re.match(r'\s{2}(.{36})\s+([0-9.]+)\s+([0-9.]+)\s+([0-9.]+)\s+([0-9.]+)', line)
        if m and current is not None:
            name = m.group(1).strip()
            if name and not name.startswith(('=', '-', 'M')):
                current[name] = float(m.group(5))  # AvgBIO
    return zs, ft

zs_res, ft_res = parse_transfer_results(path)
if ft_res:
    conditions = list(ft_res.keys())
    x = np.arange(len(conditions))
    width = 0.35

    fig, ax = plt.subplots(figsize=(9, 5))
    zs_vals = [zs_res.get(c, 0) for c in conditions]
    ft_vals = [ft_res.get(c, 0) for c in conditions]
    bars1 = ax.bar(x - width/2, zs_vals, width, label='Zero-shot',  color='#4C72B0', alpha=0.85)
    bars2 = ax.bar(x + width/2, ft_vals, width, label='Fine-tuned', color='#DD8452', alpha=0.85)

    for bar, v in zip(bars1, zs_vals):
        ax.text(bar.get_x() + bar.get_width()/2, v + 0.005, f'{v:.3f}', ha='center', fontsize=8)
    for bar, v in zip(bars2, ft_vals):
        ax.text(bar.get_x() + bar.get_width()/2, v + 0.005, f'{v:.3f}', ha='center', fontsize=8)

    ax.set_xticks(x)
    ax.set_xticklabels(conditions, fontsize=9, rotation=15, ha='right')
    ax.set_ylabel('AvgBIO')
    ax.set_title('PBMC-68K → PBMC-3K Transfer: Zero-shot vs Fine-tuned AvgBIO', fontsize=11)
    ax.legend()
    ax.set_ylim(0, max(ft_vals + zs_vals) * 1.2 + 0.05)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.yaxis.grid(True, linestyle='--', alpha=0.4)
    ax.set_axisbelow(True)
    plt.tight_layout()
    plt.savefig('transfer_pbmc68k_results.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved transfer_pbmc68k_results.png')

In [ ]:
# Cell 17: Save PBMC-68K transfer results to Drive
import shutil, os

RESULTS_DIR = '/content/drive/MyDrive/CellJEPA_results/pbmc68k_pretrain/'
files = [
    'results_transfer_pbmc68k.txt',
    'results_transfer_pbmc68k_smoke.txt',
    'transfer_pbmc68k_results.png',
]
for f in files:
    if os.path.exists(f):
        shutil.copy(f, RESULTS_DIR)
        print(f'Copied {f}')
    else:
        print(f'Not found: {f} (skipping)')
print(f'Done. Files in {RESULTS_DIR}')

In [ ]:
# Cell 18: SIGReg w_sigreg=0 test — smoke test (~5 min CPU)
import subprocess, os

REPO_DIR      = '/content'
KIDNEY_CKPT   = '/content/drive/MyDrive/CellJEPA_results/kidney_sigreg/kidney_sigreg_final.pt'
KIDNEY_GENES  = '/content/drive/MyDrive/CellJEPA_results/kidney_sigreg/kidney_sigreg_gene_names.json'
PBMC68K_CKPT  = '/content/drive/MyDrive/CellJEPA_results/pbmc68k_pretrain/pbmc68k_sigreg_final.pt'
PBMC68K_GENES = '/content/drive/MyDrive/CellJEPA_results/pbmc68k_pretrain/pbmc68k_gene_names.json'

result = subprocess.run(
    ['python3', '-u', os.path.join(REPO_DIR, 'run_sigreg_w0_test.py'),
     '--smoke_test', '--device', 'cpu',
     '--kidney_checkpoint',  KIDNEY_CKPT,
     '--kidney_genes',       KIDNEY_GENES,
     '--pbmc68k_checkpoint', PBMC68K_CKPT,
     '--pbmc68k_genes',      PBMC68K_GENES,
     '--results_file',       'results_sigreg_w0_test_smoke.txt'],
    capture_output=True, text=True
)
print(result.stdout)
if result.stderr:
    print('STDERR:', result.stderr[-3000:])

In [ ]:
# Cell 19: SIGReg w_sigreg=0 test — full run (~30 min on A100)
# Tests whether disabling SIGReg loss during fine-tuning recovers AvgBIO
# for kidney and PBMC-68K pre-trained checkpoints.
import subprocess, time, threading, os

REPO_DIR      = '/content'
KIDNEY_CKPT   = '/content/drive/MyDrive/CellJEPA_results/kidney_sigreg/kidney_sigreg_final.pt'
KIDNEY_GENES  = '/content/drive/MyDrive/CellJEPA_results/kidney_sigreg/kidney_sigreg_gene_names.json'
PBMC68K_CKPT  = '/content/drive/MyDrive/CellJEPA_results/pbmc68k_pretrain/pbmc68k_sigreg_final.pt'
PBMC68K_GENES = '/content/drive/MyDrive/CellJEPA_results/pbmc68k_pretrain/pbmc68k_gene_names.json'

t0 = time.time()
proc = subprocess.Popen(
    ['python3', '-u', os.path.join(REPO_DIR, 'run_sigreg_w0_test.py'),
     '--device', 'cuda',
     '--finetune_epochs', '30',
     '--kidney_checkpoint',  KIDNEY_CKPT,
     '--kidney_genes',       KIDNEY_GENES,
     '--pbmc68k_checkpoint', PBMC68K_CKPT,
     '--pbmc68k_genes',      PBMC68K_GENES,
     '--results_file',       'results_sigreg_w0_test.txt'],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True, bufsize=1
)

def stream(pipe):
    for line in pipe:
        print(line, end='', flush=True)

t_out = threading.Thread(target=stream, args=(proc.stdout,))
t_err = threading.Thread(target=stream, args=(proc.stderr,))
t_out.start(); t_err.start()
t_out.join(); t_err.join()

rc = proc.wait()
if rc != 0:
    print(f'
*** PROCESS EXITED WITH CODE {rc} ***')
else:
    print(f'
Done in {(time.time()-t0)/60:.1f} min')

In [ ]:
# Cell 20: Display w_sigreg=0 test results
import os

path = 'results_sigreg_w0_test.txt'
if os.path.exists(path):
    print(open(path).read())
else:
    print('Results file not found — run Cell 19 first')

In [ ]:
# Cell 21: Save w_sigreg=0 test results to Drive
import shutil, os

RESULTS_DIR = '/content/drive/MyDrive/CellJEPA_results/'
for f in ['results_sigreg_w0_test.txt', 'results_sigreg_w0_test_smoke.txt']:
    if os.path.exists(f):
        shutil.copy(f, RESULTS_DIR)
        print(f'Copied {f}')
    else:
        print(f'Not found: {f} (skipping)')